In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 15:17:56 WARN Utils: Your hostname, codespaces-d5eb9c, resolves to a loopback address: 127.0.0.1; using 10.0.2.119 instead (on interface eth0)
26/05/06 15:17:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 15:17:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Question 1: Install Spark and PySpark

    Install Spark
    Run PySpark
    Create a local spark session
    Execute spark.version.
    
**Solution: '4.1.1'**

In [2]:
#Solution:
spark.version

'4.1.1'

In [ ]:
df_november = spark.read.parquet("/workspaces/docker-workshop/06 - batch/homework/data/yellow_tripdata_2025-11.parquet")

Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

    6MB
    25MB
    75MB
    100MB

**Solution: 25 MB**

In [ ]:
df = df_november.repartition(4)

In [ ]:
df.write.parquet("homework/data/yellow_tripdata_2025-11-single.parquet")

Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

    62,610
    102,340
    162,604
    225,768

**Solution: 162,604**

In [ ]:
df_november.createOrReplaceTempView("trips_data")

In [ ]:
df_result = spark.sql("""
SELECT count(*) as count_trips
FROM trips_data
WHERE 
    tpep_pickup_datetime >= '2025-11-15 00:00:00' AND tpep_pickup_datetime <= '2025-11-15 23:59:59'
                    
"""                    
)

In [ ]:
df_result.show()

Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

    22.7
    58.2
    90.6
    134.5

**Solution: 90.6**

In [ ]:
from pyspark.sql import types

In [ ]:
schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True), 
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True), 
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True), 
    types.StructField("store_and_fwd_flag", types.StringType(), True), 
    types.StructField("RatecodeID", types.IntegerType(), True), 
    types.StructField("PULocationID", types.IntegerType(), True), 
    types.StructField("DOLocationID", types.IntegerType(), True), 
    types.StructField("passenger_count", types.IntegerType(), True), 
    types.StructField("trip_distance", types.DoubleType(), True), 
    types.StructField("fare_amount", types.DoubleType(), True), 
    types.StructField("extra", types.DoubleType(), True), 
    types.StructField("mta_tax", types.DoubleType(), True), 
    types.StructField("tip_amount", types.DoubleType(), True), 
    types.StructField("tolls_amount", types.DoubleType(), True), 
    types.StructField("ehail_fee", types.DoubleType(), True), 
    types.StructField("improvement_surcharge", types.DoubleType(), True), 
    types.StructField("total_amount", types.DoubleType(), True), 
    types.StructField("payment_type", types.IntegerType(), True), 
    types.StructField("trip_type", types.IntegerType(), True), 
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])


In [ ]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .parquet("homework/data/yellow_tripdata_2025-11.parquet") 

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df_result = (
    df
    .withColumn(
        "duration_hours",
        (F.col("tpep_dropoff_datetime").cast("long") - 
         F.col("tpep_pickup_datetime").cast("long")) / 3600.0
    )
    .agg(F.max("duration_hours").alias("max_duration_hours"))
)

df_result.show()

Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?
	• 80
	• 443
	• 4040
	• 8080

Solution: 4040

Question 6: Least frequent pickup location zone


Load the zone lookup data into a temp view in Spark:

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

    Governor's Island/Ellis Island/Liberty Island
    Arden Heights
    Rikers Island
    Jamaica Bay

If multiple answers are correct, select any

**Solution: Governor's Island/Ellis Island/Liberty Island & Arden Heights**

In [ ]:
zones = spark.read.csv("01 - pipeline/taxi_zone_lookup.csv", header=True)

In [ ]:
zones.printSchema()

In [ ]:
zones.createOrReplaceTempView("zones")

In [ ]:
df_result = spark.sql(""" 
select z.Zone, count(*) as count_trips
from trips_data t
join zones z on t.PULocationID = z.LocationID
group by z.Zone
order by count_trips asc
""").show()